<a href="https://colab.research.google.com/github/tracyxoxo/TCC-IA/blob/dev/TCC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Dados - Trabalho de Conclusão de Curso

#Este notebook reúne as etapas de preparação, análise e visualização dos dados utilizados no desenvolvimento do TCC, servindo como base para a construção e avaliação dos modelos propostos ao longo do trabalho.


## Configurações do webcrawler
## Por se tratar de um ambiente linux, rodando o jupyter notebok, é necessário:

## 1. Instalar o google chrome
## 2. Instalar chrome driver
## 3. Instalar a biblioteca selenium

In [ ]:
# Instalação do google chrome
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -fy install

#Instalação e verificacao da ultima versão do chromedriver
!LATEST=$(curl -s https://googlechromelabs.github.io/chrome-for-testing/LATEST_RELEASE_STABLE) && \
wget -O chromedriver_linux64.zip "https://storage.googleapis.com/chrome-for-testing-public/${LATEST}/linux64/chromedriver-linux64.zip" && \
unzip -o chromedriver_linux64.zip && \
mv -f chromedriver-linux64/chromedriver /usr/bin/chromedriver && \
chmod +x /usr/bin/chromedriver

# Instalação da bilbioteca para o webcrawler
!pip install -q selenium

## **WebCrawler para capturar dados historicos do INMET.**

## **Configurado para executar o webcrawler no modo "headless" (sem interface)**

In [ ]:
# Biblioteca selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Bibliotecas para manipulação webcrawler
import time
import os
import requests

# Configurações iniciais do google chrome
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.binary_location = "/usr/bin/google-chrome"

driver = webdriver.Chrome(service=Service("/usr/bin/chromedriver"), options=chrome_options)

url = "https://portal.inmet.gov.br/dadoshistoricos"
driver.get(url)
time.sleep(3)

save_folder = "/content/inmet_files"
os.makedirs(save_folder, exist_ok=True)

def get_all_article_links(driver):
    links = []
    index = 1
    while True:
        try:
            xpath = f"/html/body/div[4]/div/div/article[{index}]/a"
            element = driver.find_element("xpath", xpath)
            href = element.get_attribute("href")
            print(f"Found link #{index}: {href}")
            links.append(href)
            index += 1
        except Exception as e:
            print(f"Parou no article {index}, erro: {e}")
            break
    return links

all_links = get_all_article_links(driver)

for i, link in enumerate(all_links):
    print(f"Baixando arquivo {i+1} de {len(all_links)}: {link}")
    response = requests.get(link)
    if response.status_code == 200:
        filename = link.split("/")[-1].split("?")[0]
        path = os.path.join(save_folder, filename)
        with open(path, "wb") as f:
            f.write(response.content)
        print(f"Salvo em {path}")
    else:
        print(f"Erro ao baixar {link}: Status {response.status_code}")

driver.quit()

# Processamento dos dados

### **Bibliotecas da aplicação**

In [ ]:
# Baixando diretamente do repositório o arquivo utilities.py
# !wget https://raw.githubusercontent.com/tracyxoxo/TCC-IA/refs/heads/dev/utilities.py

# Bibliotecas da aplicação
import pandas as pd
import matplotlib.pyplot as plt
from shutil import move
from utilities import extract_filtered_files,standard_hour,create_incorrect_data_matrix, plot_two_variables_vs_time
from datetime import time
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

## **Função proveniente do arquivo utilities para filtro dos arquivos com base na torre**

In [ ]:
# filtered_files = extract_filtered_files(
#     files_path=r"/content/inmet_files",
#     extract_path=r"/data_extracted",
#     pattern='INMET_SE_SP_A705_BAURU'
# )

# filtered_files

filtered_files = []
for root, dirs, files in os.walk("/home/dolores/Documents/matheus-ferreira/TCC-IA/data/_extracted_files"):
    for file in files:
        if "INMET_SE_SP_A705_BAURU" in file:
            filterFiles_path = os.path.join(root, file)
            filtered_files.append(filterFiles_path)

## **Processando dados dos diferentes anos disponíveis no website do INMET com base na torre selecionada (BAURU - SP) (2001 - 2025)**

In [ ]:
df_list = []

files = filtered_files

for file in files:
    df = pd.read_csv(file, encoding="ISO-8859-1", sep=";", skiprows=8, header=0)
    df = df.rename(columns={
        'DATA (YYYY-MM-DD)': 'Data',
        'HORA (UTC)' : 'Hora UTC',
        'RADIACAO GLOBAL (KJ/m²)' : 'RADIACAO GLOBAL (Kj/m²)'
    })
    df_list.append(df)
    general_df = pd.concat(df_list, ignore_index=True)

In [ ]:
general_df

## **Removendo coluna vazia**



In [ ]:
general_df.drop(['Unnamed: 19'], axis=1, inplace=True)

## **Verificando os tipos iniciais de cada coluna**

In [ ]:
general_df.dtypes

## Descrição das colunas
**PRECIPITAÇÃO TOTAL, HORÁRIO (mm)** – Quantidade total de chuva acumulada em um intervalo de uma hora, medida em milímetros (mm).    

**PRESSÃO ATMOSFÉRICA AO NÍVEL DA ESTAÇÃO, HORÁRIA (mB)** – Pressão atmosférica medida no local da estação meteorológica em milibares (mB).  

**PRESSÃO ATMOSFÉRICA NA HORA ANT. (AUT) (mB)** – Valor da pressão atmosférica registrado na hora anterior.    

**RADIAÇÃO GLOBAL (Kj/m²)** – Quantidade total de energia solar recebida por metro quadrado de superfície em uma hora, medida em kilojoules por metro quadrado (Kj/m²).    

**TEMPERATURA DO AR - BULBO SECO, HORÁRIA (°C)** – Temperatura do ar medida por um termômetro comum (sem influência da umidade), expressa em graus Celsius (°C).  

**TEMPERATURA DO PONTO DE ORVALHO (°C)** – Temperatura na qual o ar se torna saturado de umidade e ocorre a condensação, formando orvalho.  

**TEMPERATURA NA HORA ANT. (AUT) (°C)** – Temperatura do ar registrada na hora anterior.  

**TEMPERATURA ORVALHO NA HORA ANT. (AUT) (°C)** – Temperatura do ponto de orvalho registrada na hora anterior.  

**UMIDADE REL. NA HORA ANT. (AUT) (%)** – Umidade relativa do ar registrada na hora anterior.  

**UMIDADE RELATIVA DO AR, HORÁRIA (%)** – Quantidade de vapor d'água presente no ar em relação à quantidade máxima que ele pode conter a uma determinada temperatura, expressa em porcentagem.  

**VENTO, DIREÇÃO HORÁRIA (gr) (° (gr))** – Direção média do vento ao longo da última hora, medida em graus (°) em relação ao norte.  

**VENTO, RAJADA MÁXIMA (m/s)** – Maior velocidade do vento em um curto intervalo de tempo dentro da última hora, medida em metros por segundo (m/s).  

**VENTO, VELOCIDADE HORÁRIA (m/s)** – Velocidade média do vento ao longo da última hora, medida em metros por segundo (m/s).



### **Conversão do tipos de cada coluna**

In [ ]:
general_df['Data'] = general_df['Data'].str.replace('-','/')
general_df['Data'] = pd.to_datetime(general_df['Data'])

general_df['Hora UTC'] = general_df['Hora UTC'].apply(standard_hour)
general_df['Hora UTC'] = pd.to_datetime(general_df['Hora UTC'], format='%H%M').dt.time

general_df['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'] = general_df['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'].str.replace(',', '.')
general_df['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'] = general_df['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'].astype(float)

general_df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = general_df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].str.replace(',', '.')
general_df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = general_df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].astype(float)

general_df['PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)'] = general_df['PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)'].str.replace(',', '.')
general_df['PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)'] = general_df['PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)'].astype(float)

general_df['PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)'] = general_df['PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)'].str.replace(',','.')
general_df['PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)'] = general_df['PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)'].astype(float)

general_df['PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)'] = general_df['PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)'].str.replace(',','.')
general_df['PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)'] = general_df['PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)'].astype(float)

general_df['RADIACAO GLOBAL (Kj/m²)'] = general_df['RADIACAO GLOBAL (Kj/m²)'].str.replace(',','.')
general_df['RADIACAO GLOBAL (Kj/m²)'] = general_df['RADIACAO GLOBAL (Kj/m²)'].astype(float)

general_df['TEMPERATURA DO PONTO DE ORVALHO (°C)'] = general_df['TEMPERATURA DO PONTO DE ORVALHO (°C)'].str.replace(',','.')
general_df['TEMPERATURA DO PONTO DE ORVALHO (°C)'] = general_df['TEMPERATURA DO PONTO DE ORVALHO (°C)'].astype(float)

general_df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'].str.replace(',','.')
general_df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'].astype(float)

general_df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'].str.replace(',','.')
general_df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'].astype(float)

general_df['TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)'].str.replace(',','.')
general_df['TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)'].astype(float)

general_df['TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)'].str.replace(',','.')
general_df['TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)'] = general_df['TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)'].astype(float)

general_df['VENTO, RAJADA MAXIMA (m/s)'] = general_df['VENTO, DIREÇÃO HORARIA (gr) (° (gr))'].astype(float)

general_df['VENTO, VELOCIDADE HORARIA (m/s)'] = general_df['VENTO, VELOCIDADE HORARIA (m/s)'].str.replace(',','.')
general_df['VENTO, VELOCIDADE HORARIA (m/s)'] = general_df['VENTO, VELOCIDADE HORARIA (m/s)'].astype(float)

### **Colunas convertidas para criação dos gráficos**

In [ ]:
general_df.info()

## **Analisando o por quê RADIACAO GLOBAL tem um padrão de valores nulos**

In [ ]:
radiacao_nan_df = general_df[general_df['RADIACAO GLOBAL (Kj/m²)'].isnull()]
na_por_hora = radiacao_nan_df['Hora UTC'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
na_por_hora.plot(kind='bar', color='orange')
plt.title('Frequência de Nulls por horário na radiação solar')
plt.xlabel('Horário')
plt.ylabel('Número de Nulls')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
# plt.show()
plt.savefig("fig1.png")

### Removendo os registros que tem RADIACAO GLOBAL nula e hora entre 6 da manhã e 20 da noite

In [ ]:
inicio = time(6, 0)
fim = time(20, 0)

filtro = (general_df['RADIACAO GLOBAL (Kj/m²)'].isnull()) & (general_df['Hora UTC'] >= inicio) & (general_df['Hora UTC'] < fim)

general_df = general_df[~filtro]

general_df.isnull().sum()

In [ ]:
general_df['RADIACAO GLOBAL (Kj/m²)'] = general_df['RADIACAO GLOBAL (Kj/m²)'].fillna(0)

general_df.isnull().sum()

In [ ]:
general_df.info()

In [ ]:
general_df

In [ ]:
general_df.describe()

## **Concatenando 'Data' com 'HORA UTC'**

In [ ]:
general_df['datetime'] = general_df.apply(
    lambda row: pd.Timestamp.combine(row['Data'],row['Hora UTC']),
    axis = 1
)

In [ ]:
general_df['datetime']

## Criando uma flag para dados incorretos

### Matriz de gráficos do tipo scatter com as variáveis definidas

In [ ]:
fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[
        "Temperatura",
        "Pressão Atmosférica",
        "Umidade Relativa",
        "Velocidade do Vento",
        "Precipitação",
        ""
    ]
)

create_incorrect_data_matrix(general_df, 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)', -50, 60, "Temperatura", row=1, col=1, fig=fig)
create_incorrect_data_matrix(general_df, 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)', 0, 2000, "Pressão atmosférica", row=1, col=2, fig=fig)
create_incorrect_data_matrix(general_df, 'UMIDADE RELATIVA DO AR, HORARIA (%)', 0, 300, "Umidade relativa do ar", row=2, col=1, fig=fig)
create_incorrect_data_matrix(general_df, 'VENTO, VELOCIDADE HORARIA (m/s)', 0, 1000, "Velocidade do vento", row=2, col=2, fig=fig)
create_incorrect_data_matrix(general_df, 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 0, 1000, "Precipitação", row=3, col=1, fig=fig)


fig.update_layout(
    height=1200,
    width=1000,
    title_text="Variáveis Meteorológicas com Anomalias",
    legend_title="Dados incorretos [Contagem]"
)

# fig.show()
_ = fig.to_html("fig2.html")

### Quantidade de dados incorretos (-9999)
* Temperatura x Tempo: 11%
* Pressão x Tempo: 11%
* Umidade x Tempo: 12.3%
* Vento x Tempo: 11.2%
* Precipitação x tempo: 11.2%




In [ ]:
plot_two_variables_vs_time(
    general_df,
    var1='TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
    var2='UMIDADE RELATIVA DO AR, HORARIA (%)',
    var1_label='Temperatura (°C)',
    var2_label='Umidade Relativa (%)',
    title='Temperatura e Umidade ao longo do tempo'
)

In [ ]:
general_df

In [ ]:
outlier_cols = [col for col in general_df.columns if col.startswith("is_outlier_")]
outlier_rows = general_df[general_df[outlier_cols].any(axis=1)]

outlier_rows

In [ ]:
df_no_outliers = general_df.drop(outlier_rows.index)
df_no_outliers

In [ ]:
# filtrar um subset menor com variáveis que não tem NaN values

#df_experiment_no_outliers = df_no_outliers.drop[]

df_no_outliers.isna().sum()

In [ ]:
df_experiment_training = df_no_outliers.copy()

In [ ]:
df_experiment_training['RADIACAO GLOBAL (Kj/m²)'] = df_experiment_training['RADIACAO GLOBAL (Kj/m²)'].fillna(0)

In [ ]:
df_experiment_training

In [ ]:
df_experiment_training['year'] = df_experiment_training['datetime'].dt.year
df_experiment_training['month'] = df_experiment_training['datetime'].dt.month
df_experiment_training['day'] = df_experiment_training['datetime'].dt.day

df_experiment_training['hour'] = df_experiment_training['datetime'].dt.hour

In [ ]:
df_experiment_training.columns

In [ ]:
df_experiment_training.drop(columns=['Data', 'Hora UTC','datetime','PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
                                     'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)','TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
                                     'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)','PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
                                     'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)', 'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
                                     'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)','is_outlier_TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
                                     'is_outlier_PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
                                      'is_outlier_UMIDADE RELATIVA DO AR, HORARIA (%)',
                                      'is_outlier_VENTO, VELOCIDADE HORARIA (m/s)',
                                      'is_outlier_PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'], inplace=True)

In [ ]:
df_experiment_training.columns

In [ ]:
df_general = df_experiment_training.dropna()

df_general

In [ ]:
df_general_without_date = df_general.drop(columns=['year', 'month','day','hour'])

In [ ]:
df_general_without_date.columns

# Matriz de correlação

In [ ]:
corr_matrix = df_general_without_date.corr()

corr_matrix

In [ ]:
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Correlation Matrix"
)

fig.update_layout(
    width=1600,
    height=1600
)

fig.update_xaxes(side="bottom", fixedrange=False)
fig.update_yaxes(fixedrange=False)

# fig.show()
_ = fig.to_html("fig3-corr-matrix.html")

# Análise

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

## Análise com apenas variáveis temporais

In [ ]:
x_temporal = df_general[['year', 'month', 'day', 'hour']]
y = df_general['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)']

x_temporal_train, x_temporal_test, y_temporal_train, y_temporal_test = train_test_split(x_temporal, y, test_size=0.2, random_state=42)

## Análise com todas as variáveis

In [ ]:
x = df_general.drop(columns=['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'])


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor()
xgb_model.fit(x_train, y_train)


xgb_temporal_model = XGBRegressor()
xgb_temporal_model.fit(x_temporal_train, y_temporal_train)

##R2 - xgboost

In [ ]:
xgboost_r2_model = xgb_model.score(x_test, y_test)
xgboost_r2_model

In [ ]:
xgboost_r2_temporal = xgb_temporal_model.score(x_temporal_test, y_temporal_test)
xgboost_r2_temporal

### Resultado execução local xgboost (R2)

### R2 XGBoost: - 0.9972770927100549

### R2 XGBoost Temporal - - 0.9972770927100549

## MSE - xgboost

In [ ]:
y_pred_xgb_temporal = xgb_temporal_model.predict(x_temporal_test)

y_pred_xgb = pd.DataFrame(xgb_model.predict(x_test))

xgb_mse_temporal = mean_squared_error(y_temporal_test, y_pred_xgb_temporal)
xgb_mse = mean_squared_error(y_test, y_pred_xgb)

print("MSE XGBoost Temporal:", xgb_mse_temporal)
print("MSE XGBoost:", xgb_mse)

### Resultado execução local xgboost (MSE)

### MSE XGBoost Temporal: 5.404767982078238
### MSE XGBoost: 0.06950278063492658

## RMSE - xgboost

In [ ]:
xgb_rmse_temporal = np.sqrt(xgb_mse_temporal)
xgb_rmse = np.sqrt(xgb_mse)

print("RMSE XGBoost Temporal:", xgb_rmse_temporal)
print("RMSE XGBoost:", xgb_rmse)

### Resultado execução local xgboost (RMSE)

### RMSE XGBoost Temporal: 2.3248156877649975
### RMSE XGBoost: 0.2636338002512701

## MAE - xgboost

In [ ]:
xgb_mae_temporal = mean_absolute_error(y_temporal_test, y_pred_xgb_temporal)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)

print("MAE XGBoost Temporal:", xgb_mae_temporal)
print("MAE XGBoost:", xgb_mae)

### Resultado execução local xgboost (MAE)

### MAE XGBoost Temporal: 1.7673724324576627
### MAE XGBoost: 0.17449698644129313

# Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_temporal_model = RandomForestRegressor()
rf_temporal_model.fit(x_temporal_train, y_temporal_train)

rf_model = RandomForestRegressor()
rf_model.fit(x_train, y_train)

##R2 - Random Forest

In [ ]:
r2_rf_temporal = rf_temporal_model.score(x_temporal_test, y_temporal_test)

r2_rf_temporal

In [ ]:
r2_rf_model = rf_model.score(x_test, y_test)

r2_rf_model

### Resultado execução local Random Forest (R2)
### R2 Random Forest Temporal: 0.8687601307053557
### R2 Random Forest: 0.9958673195303077

## MSE - Random Forest

In [ ]:
y_pred_rf_temporal = rf_temporal_model.predict(x_temporal_test)

y_pred_rf = pd.DataFrame(rf_model.predict(x_test))

rf_mse_temporal = mean_squared_error(y_temporal_test, y_pred_rf_temporal)
rf_mse = mean_squared_error(y_test, y_pred_rf)

print("MSE Random Forest Temporal:", rf_mse_temporal)
print("MSE Random Forest:", rf_mse)

### Resultado execução local Random Forest (MSE)
### MSE Random Forest Temporal: 3.349925236098021
### MSE Random Forest: 0.10548753722902923

## RMSE - Random Forest

In [ ]:
rf_rmse_temporal = np.sqrt(rf_mse_temporal)
rf_rmse = np.sqrt(rf_mse)

print("RMSE Random Forest Temporal:", rf_rmse_temporal)
print("RMSE Random Forest:", rf_rmse)

### Resultado execução local Random Forest (RMSE)
### RMSE Random Forest Temporal: 1.830280097716746
### RMSE Random Forest: 0.32478844996247824

## MAE - Random Forest

In [ ]:
rf_mae_temporal = mean_absolute_error(y_temporal_test, y_pred_rf_temporal)
rf_mae = mean_absolute_error(y_test, y_pred_rf)

print("MAE XGBoost Temporal:", rf_mae_temporal)
print("MAE XGBoost:", rf_mae)

### Resultado execução local Random Forest (MAE)
### MAE XGBoost Temporal: 1.300318096135721
### MAE XGBoost: 0.20564354382657865

# Support Vector Regression

In [ ]:
from sklearn.svm import SVR

In [ ]:
svr_temporal_model = SVR()
svr_temporal_model.fit(x_temporal_train, y_temporal_train)

svr_model = SVR()
svr_model.fit(x_train, y_train)

## R2 - Support Vector Regression

In [ ]:
r2_svr_temporal = svr_temporal_model.score(x_temporal_test, y_temporal_test)

r2_svr_temporal

In [ ]:
r2_svr_model = svr_model.score(x_test, y_test)

r2_svr_model

### Resultado execução local Support Vector Regression (R2)
### R2 Support Vector Regression temporal: 0.1455761345095169
### R2 Support Vector Regression: 0.47064048713611417

## MSE - Support Vector Regression

In [ ]:
y_pred_svr_temporal = svr_temporal_model.predict(x_temporal_test)

y_pred_svr = pd.DataFrame(svr_model.predict(x_test))

svr_mse_temporal = mean_squared_error(y_temporal_test, y_pred_svr_temporal)
svr_mse = mean_squared_error(y_test, y_pred_svr)

print("MSE Support Vector Temporal:", svr_mse_temporal)
print("MSE Support Vector:", svr_mse)

### Resultado execução local Support Vector Regression (MSE)
### MSE Support Vector Temporal: 21.809348673648792
### MSE Support Vector: 13.512012779668723

## RMSE - Support Vector Regression

In [ ]:
svr_rmse_temporal = np.sqrt(svr_mse_temporal)
svr_rmse = np.sqrt(svr_mse)

print("RMSE Random Forest Temporal:", svr_rmse_temporal)
print("RMSE Random Forest:", svr_rmse)

### Resultado execução local Support Vector Regression (RMSE)

### RMSE Random Forest Temporal: 4.670048037616828
### RMSE Random Forest: 3.675868982930257

## MAE - Support Vector Regression

In [ ]:
svr_mae_temporal = mean_absolute_error(y_temporal_test, y_pred_svr_temporal)
svr_mae = mean_absolute_error(y_test, y_pred_svr)

print("MAE XGBoost Temporal:", svr_mae_temporal)
print("MAE XGBoost:", svr_mae)

### Resultado execução local Support Vector Regression (MAE)
### MAE XGBoost Temporal: 3.689362002845445
### MAE XGBoost: 2.853000171137171

# Comparação

In [ ]:
svr_score = svr_model.score(x_test, y_test)
rf_score = rf_model.score(x_test, y_test)
xgb_score = xgb_model.score(x_test, y_test)

df_scores = pd.DataFrame({
    "Model": ["SVR", "Random Forest", "XGBoost"],
    "R2 Score": [svr_score, rf_score, xgb_score]
})

fig = px.bar(df_scores, x="Model", y="R2 Score", text="R2 Score",
             title="Model Comparison (R² on Test Set)", color="Model")
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
# fig.show()
_ = fig.to_html("fig4-comparison.html")

x = Actual = True target values

*   x = Actual = True target values
*   y = Predicted = model's predictions
*   y = Predicted = model's predictions
*   trendline = "ols" = blue regression line for each model's predictions




In [ ]:
y_pred_svr = svr_model.predict(x_test)
y_pred_rf = rf_model.predict(x_test)
y_pred_xgb = xgb_model.predict(x_test)

df_preds = pd.DataFrame({
    "Actual": y_test,
    "SVR": y_pred_svr,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb
})

df_melt = df_preds.melt(id_vars="Actual", var_name="Model", value_name="Predicted")

fig = px.scatter(df_melt, x="Actual", y="Predicted", color="Model",
                 title="Predicted vs Actual (All Models)", opacity=0.6,
                 trendline="ols")
fig.add_shape(type="line", x0=df_melt["Actual"].min(), y0=df_melt["Actual"].min(),
              x1=df_melt["Actual"].max(), y1=df_melt["Actual"].max(),
              line=dict(color="red", dash="dash"))
# fig.show()
_ = fig.to_html("fig4-comparison.html")


# Busca por melhores hiperparametros


## Resultado GridSearchCV (execução local)


In [ ]:
search_space = {
  "n_estimators": [100, 200, 500],
  "max_depth": [3, 6, 9],
  "eta": [0.1, 0.3, 0.6, 0.9],
}

from sklearn.model_selection import GridSearchCV
gs = GridSearchCV(
    estimator = xgb_temporal_model,
    param_grid = search_space,
    scoring = ["r2", "neg_root_mean_squared_error", "neg_median_absolute_error"],
    refit = "r2",
    verbose = 4
)



In [ ]:
gs.fit(x_train, y_train)

In [ ]:
gs.best_estimator_

In [ ]:
gs.best_params_

### {'eta': 0.3, 'max_depth': 6, 'n_estimators': 500}

## Random Forest - Grid search

In [ ]:
search_space_random_forest = {
  "n_estimators": [100, 200, 500],
  "max_depth": [3, 6, 9],
  "max_features": ["sqrt", "log2"],
  "min_samples_split": [2, 5, 10],
  "min_samples_leaf": [1, 2, 4]
}

gs_random_forest = GridSearchCV(
    estimator = rf_temporal_model,
    param_grid = search_space_random_forest,
    scoring = ["r2", "neg_root_mean_squared_error", "neg_median_absolute_error"],
    refit = "r2",
    cv = 3,
    verbose = 4,
    n_jobs=-1
)

In [ ]:
gs_random_forest.fit(x_train, y_train)

In [ ]:
gs_random_forest.best_estimator_

In [ ]:
gs_random_forest.best_params_

### {'max_depth': 9,'max_features': 'sqrt', 'min_samples_leaf': 1 'min_samples_split': 5, 'n_estimators': 500}

## SVR - Resultado GridSearchCV (execução local)

In [ ]:
search_space_svr = {
    "kernel": ["poly"],   # tipo de kernel
    # "kernel": ["linear", "sigmoid"],   # tipo de kernel
    "degree": [2, 3, 4],               # grau do polinômio (usado só quando kernel="poly")
    "gamma": ["auto"],        # parâmetro do kernel (rbf, poly, sigmoid)
    "tol": [1e-3, 1e-4],               # tolerância para critério de parada
    "C": [0.1, 1],                     # regularização (quanto maior, mais complexo o modelo)
    "epsilon": [0.01, 0.1]             # margem de insensibilidade
}

gs_svr = GridSearchCV(
    estimator = svr_temporal_model,
    param_grid = search_space_svr,
    scoring = ["r2", "neg_root_mean_squared_error", "neg_median_absolute_error"],
    refit = "r2",
    verbose = 4,
    n_jobs=24,
    cv=3
)



In [ ]:
gs_svr.fit(x_train, y_train)

In [ ]:
gs_svr.best_estimator_

# Melhores dados

- 0.954;C=0.1, epsilon=0.1 , gamma=auto, kernel=linear, tol=0.0001
- 0.955;C=0.1, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001;
- 0.955;C=0.1, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001;
- 0.955;C=0.1, epsilon=0.1 , gamma=auto, kernel=linear, tol=0.0001;
- 0.955;C=0.1, epsilon=0.1 , gamma=auto, kernel=linear, tol=0.001;



# Todos os dados
[CV 1/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.695) neg_root_mean_squared_error: (test=-1.181) r2: (test=0.945) total time=767.7min
[CV 1/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.717) neg_root_mean_squared_error: (test=-1.171) r2: (test=0.946) total time=809.3min
[CV 1/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.510) neg_root_mean_squared_error: (test=-1458.790) r2: (test=-84367.489) total time=61.3min
[CV 1/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.510) neg_root_mean_squared_error: (test=-1458.790) r2: (test=-84367.489) total time=61.3min
[CV 1/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.725) neg_root_mean_squared_error: (test=-1.239) r2: (test=0.939) total time=1097.2min
[CV 1/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.625) neg_root_mean_squared_error: (test=-1.099) r2: (test=0.952) total time=704.3min
[CV 1/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.500) neg_root_mean_squared_error: (test=-1458.791) r2: (test=-84367.615) total time=34.3min
[CV 1/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.500) neg_root_mean_squared_error: (test=-1458.791) r2: (test=-84367.615) total time=34.5min
[CV 1/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.695) neg_root_mean_squared_error: (test=-1.181) r2: (test=0.945) total time=748.9min
[CV 1/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.717) neg_root_mean_squared_error: (test=-1.171) r2: (test=0.946) total time=804.4min
[CV 1/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.510) neg_root_mean_squared_error: (test=-1458.790) r2: (test=-84367.489) total time=57.6min
[CV 1/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.510) neg_root_mean_squared_error: (test=-1458.790) r2: (test=-84367.489) total time=57.6min
[CV 1/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.725) neg_root_mean_squared_error: (test=-1.239) r2: (test=0.939) total time=1065.7min
[CV 1/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.625) neg_root_mean_squared_error: (test=-1.099) r2: (test=0.952) total time=681.1min
[CV 1/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.500) neg_root_mean_squared_error: (test=-1458.791) r2: (test=-84367.615) total time=13.8min
[CV 1/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.500) neg_root_mean_squared_error: (test=-1458.791) r2: (test=-84367.615) total time=12.8min
[CV 1/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.695) neg_root_mean_squared_error: (test=-1.181) r2: (test=0.945) total time=768.2min
[CV 1/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.717) neg_root_mean_squared_error: (test=-1.171) r2: (test=0.946) total time=802.3min
[CV 1/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.510) neg_root_mean_squared_error: (test=-1458.790) r2: (test=-84367.489) total time=42.1min
[CV 1/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.510) neg_root_mean_squared_error: (test=-1458.790) r2: (test=-84367.489) total time=39.2min
[CV 1/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.725) neg_root_mean_squared_error: (test=-1.239) r2: (test=0.939) total time=1084.4min
[CV 1/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.625) neg_root_mean_squared_error: (test=-1.099) r2: (test=0.952) total time=700.3min
[CV 1/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.500) neg_root_mean_squared_error: (test=-1458.791) r2: (test=-84367.615) total time=25.1min
[CV 1/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.500) neg_root_mean_squared_error: (test=-1458.791) r2: (test=-84367.615) total time=24.2min
[CV 1/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-2.678) neg_root_mean_squared_error: (test=-8.322) r2: (test=-1.745) total time=878.2min
[CV 1/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.787) neg_root_mean_squared_error: (test=-3.610) r2: (test=0.483) total time=840.9min
[CV 1/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-44.610) neg_root_mean_squared_error: (test=-14593.950) r2: (test=-8443845.031) total time=25.2min
[CV 1/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-44.610) neg_root_mean_squared_error: (test=-14593.950) r2: (test=-8443845.031) total time=27.0min
[CV 1/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.292) neg_root_mean_squared_error: (test=-1.908) r2: (test=0.856) total time=872.3min
[CV 1/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.508) neg_root_mean_squared_error: (test=-2.663) r2: (test=0.719) total time=918.8min
[CV 1/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-44.613) neg_root_mean_squared_error: (test=-14593.948) r2: (test=-8443841.906) total time=28.9min
[CV 1/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-44.613) neg_root_mean_squared_error: (test=-14593.948) r2: (test=-8443841.906) total time=25.2min
[CV 1/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-2.678) neg_root_mean_squared_error: (test=-8.322) r2: (test=-1.745) total time=830.9min
[CV 1/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.787) neg_root_mean_squared_error: (test=-3.610) r2: (test=0.483) total time=804.9min
[CV 1/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-44.610) neg_root_mean_squared_error: (test=-14593.950) r2: (test=-8443845.031) total time=20.6min
[CV 1/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-44.610) neg_root_mean_squared_error: (test=-14593.950) r2: (test=-8443845.031) total time=19.5min
[CV 1/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.292) neg_root_mean_squared_error: (test=-1.908) r2: (test=0.856) total time=827.4min
[CV 1/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.508) neg_root_mean_squared_error: (test=-2.663) r2: (test=0.719) total time=854.0min
[CV 1/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-44.613) neg_root_mean_squared_error: (test=-14593.948) r2: (test=-8443841.906) total time=27.1min
[CV 1/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-44.613) neg_root_mean_squared_error: (test=-14593.948) r2: (test=-8443841.906) total time=36.0min
[CV 1/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-2.678) neg_root_mean_squared_error: (test=-8.322) r2: (test=-1.745) total time=729.3min
[CV 1/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.787) neg_root_mean_squared_error: (test=-3.610) r2: (test=0.483) total time=727.7min
[CV 1/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-44.610) neg_root_mean_squared_error: (test=-14593.950) r2: (test=-8443845.031) total time=16.8min
[CV 1/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-44.610) neg_root_mean_squared_error: (test=-14593.950) r2: (test=-8443845.031) total time=19.9min
[CV 1/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.292) neg_root_mean_squared_error: (test=-1.908) r2: (test=0.856) total time=645.4min
[CV 1/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.508) neg_root_mean_squared_error: (test=-2.663) r2: (test=0.719) total time=690.5min
[CV 1/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-44.613) neg_root_mean_squared_error: (test=-14593.948) r2: (test=-8443841.906) total time=24.4min
[CV 1/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-44.613) neg_root_mean_squared_error: (test=-14593.948) r2: (test=-8443841.906) total time=23.3min
[CV 2/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.610) neg_root_mean_squared_error: (test=-1.085) r2: (test=0.953) total time=857.8min
[CV 2/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.672) neg_root_mean_squared_error: (test=-1.152) r2: (test=0.947) total time=749.6min
[CV 2/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.310) neg_root_mean_squared_error: (test=-1459.066) r2: (test=-85448.809) total time=34.1min
[CV 2/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.310) neg_root_mean_squared_error: (test=-1459.066) r2: (test=-85448.809) total time=34.1min
[CV 2/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.605) neg_root_mean_squared_error: (test=-1.062) r2: (test=0.955) total time=755.5min
[CV 2/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.615) neg_root_mean_squared_error: (test=-1.059) r2: (test=0.955) total time=813.8min
[CV 2/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.300) neg_root_mean_squared_error: (test=-1459.065) r2: (test=-85448.698) total time=34.4min
[CV 2/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.300) neg_root_mean_squared_error: (test=-1459.065) r2: (test=-85448.698) total time=33.9min
[CV 2/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.610) neg_root_mean_squared_error: (test=-1.085) r2: (test=0.953) total time=854.8min
[CV 2/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.672) neg_root_mean_squared_error: (test=-1.152) r2: (test=0.947) total time=748.2min
[CV 2/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.310) neg_root_mean_squared_error: (test=-1459.066) r2: (test=-85448.809) total time=31.0min
[CV 2/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.310) neg_root_mean_squared_error: (test=-1459.066) r2: (test=-85448.809) total time=32.8min
[CV 2/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.605) neg_root_mean_squared_error: (test=-1.062) r2: (test=0.955) total time=719.0min
[CV 2/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.615) neg_root_mean_squared_error: (test=-1.059) r2: (test=0.955) total time=794.4min
[CV 2/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.300) neg_root_mean_squared_error: (test=-1459.065) r2: (test=-85448.698) total time=14.5min
[CV 2/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.300) neg_root_mean_squared_error: (test=-1459.065) r2: (test=-85448.698) total time=12.5min
[CV 2/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.610) neg_root_mean_squared_error: (test=-1.085) r2: (test=0.953) total time=855.2min
[CV 2/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.672) neg_root_mean_squared_error: (test=-1.152) r2: (test=0.947) total time=758.2min
[CV 2/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.310) neg_root_mean_squared_error: (test=-1459.066) r2: (test=-85448.809) total time=22.7min
[CV 2/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.310) neg_root_mean_squared_error: (test=-1459.066) r2: (test=-85448.809) total time=21.7min
[CV 2/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.605) neg_root_mean_squared_error: (test=-1.062) r2: (test=0.955) total time=745.6min
[CV 2/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.615) neg_root_mean_squared_error: (test=-1.059) r2: (test=0.955) total time=810.0min
[CV 2/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.300) neg_root_mean_squared_error: (test=-1459.065) r2: (test=-85448.698) total time=24.9min
[CV 2/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.300) neg_root_mean_squared_error: (test=-1459.065) r2: (test=-85448.698) total time=24.2min
[CV 2/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.629) neg_root_mean_squared_error: (test=-2.521) r2: (test=0.745) total time=750.9min
[CV 2/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.530) neg_root_mean_squared_error: (test=-2.528) r2: (test=0.744) total time=784.7min
[CV 2/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-41.738) neg_root_mean_squared_error: (test=-14596.535) r2: (test=-8551858.302) total time=24.9min
[CV 2/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-41.738) neg_root_mean_squared_error: (test=-14596.535) r2: (test=-8551858.302) total time=27.1min
[CV 2/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.703) neg_root_mean_squared_error: (test=-2.617) r2: (test=0.725) total time=930.2min
[CV 2/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.454) neg_root_mean_squared_error: (test=-2.160) r2: (test=0.813) total time=876.5min
[CV 2/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-41.700) neg_root_mean_squared_error: (test=-14596.557) r2: (test=-8551884.022) total time=33.3min
[CV 2/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-41.700) neg_root_mean_squared_error: (test=-14596.557) r2: (test=-8551884.022) total time=25.8min
[CV 2/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.629) neg_root_mean_squared_error: (test=-2.521) r2: (test=0.745) total time=730.5min
[CV 2/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.530) neg_root_mean_squared_error: (test=-2.528) r2: (test=0.744) total time=773.4min
[CV 2/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-41.738) neg_root_mean_squared_error: (test=-14596.535) r2: (test=-8551858.302) total time=21.8min
[CV 2/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-41.738) neg_root_mean_squared_error: (test=-14596.535) r2: (test=-8551858.302) total time=20.0min
[CV 2/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.703) neg_root_mean_squared_error: (test=-2.617) r2: (test=0.725) total time=852.4min
[CV 2/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.454) neg_root_mean_squared_error: (test=-2.160) r2: (test=0.813) total time=825.6min
[CV 2/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-41.700) neg_root_mean_squared_error: (test=-14596.557) r2: (test=-8551884.022) total time=30.8min
[CV 2/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-41.700) neg_root_mean_squared_error: (test=-14596.557) r2: (test=-8551884.022) total time=29.8min
[CV 2/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.629) neg_root_mean_squared_error: (test=-2.521) r2: (test=0.745) total time=660.7min
[CV 2/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.530) neg_root_mean_squared_error: (test=-2.528) r2: (test=0.744) total time=704.5min
[CV 2/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-41.738) neg_root_mean_squared_error: (test=-14596.535) r2: (test=-8551858.302) total time=16.5min
[CV 2/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-41.738) neg_root_mean_squared_error: (test=-14596.535) r2: (test=-8551858.302) total time=18.9min
[CV 2/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.703) neg_root_mean_squared_error: (test=-2.617) r2: (test=0.725) total time=573.2min
[CV 2/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.454) neg_root_mean_squared_error: (test=-2.160) r2: (test=0.813) total time=669.2min
[CV 2/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-41.700) neg_root_mean_squared_error: (test=-14596.557) r2: (test=-8551884.022) total time=23.5min
[CV 2/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-41.700) neg_root_mean_squared_error: (test=-14596.557) r2: (test=-8551884.022) total time=23.4min
[CV 3/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.617) neg_root_mean_squared_error: (test=-1.058) r2: (test=0.955) total time=775.1min
[CV 3/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.603) neg_root_mean_squared_error: (test=-1.066) r2: (test=0.955) total time=838.7min
[CV 3/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.110) neg_root_mean_squared_error: (test=-1459.242) r2: (test=-84898.820) total time=34.1min
[CV 3/3] END C=0.1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.110) neg_root_mean_squared_error: (test=-1459.242) r2: (test=-84898.820) total time=34.5min
[CV 3/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.602) neg_root_mean_squared_error: (test=-1.079) r2: (test=0.954) total time=743.9min
[CV 3/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.643) neg_root_mean_squared_error: (test=-1.089) r2: (test=0.953) total time=784.5min
[CV 3/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.100) neg_root_mean_squared_error: (test=-1459.241) r2: (test=-84898.618) total time=34.0min
[CV 3/3] END C=0.1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.100) neg_root_mean_squared_error: (test=-1459.240) r2: (test=-84898.546) total time=34.4min
[CV 3/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.617) neg_root_mean_squared_error: (test=-1.058) r2: (test=0.955) total time=765.7min
[CV 3/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.603) neg_root_mean_squared_error: (test=-1.066) r2: (test=0.955) total time=820.1min
[CV 3/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.110) neg_root_mean_squared_error: (test=-1459.242) r2: (test=-84898.820) total time=31.0min
[CV 3/3] END C=0.1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.110) neg_root_mean_squared_error: (test=-1459.242) r2: (test=-84898.820) total time=32.4min
[CV 3/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.602) neg_root_mean_squared_error: (test=-1.079) r2: (test=0.954) total time=700.5min
[CV 3/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.643) neg_root_mean_squared_error: (test=-1.089) r2: (test=0.953) total time=768.1min
[CV 3/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.100) neg_root_mean_squared_error: (test=-1459.241) r2: (test=-84898.618) total time=15.7min
[CV 3/3] END C=0.1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.100) neg_root_mean_squared_error: (test=-1459.240) r2: (test=-84898.546) total time=12.0min
[CV 3/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.617) neg_root_mean_squared_error: (test=-1.058) r2: (test=0.955) total time=780.7min
[CV 3/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.603) neg_root_mean_squared_error: (test=-1.066) r2: (test=0.955) total time=832.2min
[CV 3/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.110) neg_root_mean_squared_error: (test=-1459.242) r2: (test=-84898.820) total time=23.7min
[CV 3/3] END C=0.1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.110) neg_root_mean_squared_error: (test=-1459.242) r2: (test=-84898.820) total time=21.7min
[CV 3/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-0.602) neg_root_mean_squared_error: (test=-1.079) r2: (test=0.954) total time=730.3min
[CV 3/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-0.643) neg_root_mean_squared_error: (test=-1.089) r2: (test=0.953) total time=788.6min
[CV 3/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-7.100) neg_root_mean_squared_error: (test=-1459.241) r2: (test=-84898.618) total time=25.1min
[CV 3/3] END C=0.1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-7.100) neg_root_mean_squared_error: (test=-1459.240) r2: (test=-84898.546) total time=23.9min
[CV 3/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.539) neg_root_mean_squared_error: (test=-5.204) r2: (test=-0.080) total time=910.2min
[CV 3/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.244) neg_root_mean_squared_error: (test=-1.974) r2: (test=0.845) total time=868.2min
[CV 3/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-33.210) neg_root_mean_squared_error: (test=-14602.629) r2: (test=-8501859.948) total time=24.7min
[CV 3/3] END C=1, degree=2, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-33.210) neg_root_mean_squared_error: (test=-14602.629) r2: (test=-8501859.948) total time=26.5min
[CV 3/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.569) neg_root_mean_squared_error: (test=-2.413) r2: (test=0.768) total time=880.7min
[CV 3/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.787) neg_root_mean_squared_error: (test=-3.482) r2: (test=0.516) total time=943.5min
[CV 3/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-33.200) neg_root_mean_squared_error: (test=-14602.636) r2: (test=-8501867.994) total time=35.3min
[CV 3/3] END C=1, degree=2, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-33.200) neg_root_mean_squared_error: (test=-14602.636) r2: (test=-8501867.994) total time=28.1min
[CV 3/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.539) neg_root_mean_squared_error: (test=-5.204) r2: (test=-0.080) total time=851.0min
[CV 3/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.244) neg_root_mean_squared_error: (test=-1.974) r2: (test=0.845) total time=838.8min
[CV 3/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-33.210) neg_root_mean_squared_error: (test=-14602.629) r2: (test=-8501859.948) total time=22.2min
[CV 3/3] END C=1, degree=3, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-33.210) neg_root_mean_squared_error: (test=-14602.629) r2: (test=-8501859.948) total time=20.4min
[CV 3/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.569) neg_root_mean_squared_error: (test=-2.413) r2: (test=0.768) total time=843.4min
[CV 3/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.787) neg_root_mean_squared_error: (test=-3.482) r2: (test=0.516) total time=848.0min
[CV 3/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-33.200) neg_root_mean_squared_error: (test=-14602.636) r2: (test=-8501867.994) total time=31.6min
[CV 3/3] END C=1, degree=3, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-33.200) neg_root_mean_squared_error: (test=-14602.636) r2: (test=-8501867.994) total time=27.2min
[CV 3/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.539) neg_root_mean_squared_error: (test=-5.204) r2: (test=-0.080) total time=727.3min
[CV 3/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.244) neg_root_mean_squared_error: (test=-1.974) r2: (test=0.845) total time=721.5min
[CV 3/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-33.210) neg_root_mean_squared_error: (test=-14602.629) r2: (test=-8501859.948) total time=16.0min
[CV 3/3] END C=1, degree=4, epsilon=0.01, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-33.210) neg_root_mean_squared_error: (test=-14602.629) r2: (test=-8501859.948) total time=18.9min
[CV 3/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.0001; neg_median_absolute_error: (test=-1.569) neg_root_mean_squared_error: (test=-2.413) r2: (test=0.768) total time=568.8min
[CV 3/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=linear, tol=0.001; neg_median_absolute_error: (test=-1.787) neg_root_mean_squared_error: (test=-3.482) r2: (test=0.516) total time=684.3min
[CV 3/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.0001; neg_median_absolute_error: (test=-33.200) neg_root_mean_squared_error: (test=-14602.636) r2: (test=-8501867.994) total time=22.6min
[CV 3/3] END C=1, degree=4, epsilon=0.1, gamma=auto, kernel=sigmoid, tol=0.001; neg_median_absolute_error: (test=-33.200) neg_root_mean_squared_error: (test=-14602.636) r2: (test=-8501867.994) total time=25.2min


In [ ]:
gs_svr.best_params_